# Eurostat API, metadata and data preparation

Build the 2024 EU NUTS 2 analysis table from five preserved API responses.
Run this notebook from top to bottom with the project `.venv` kernel.
The default workflow is offline: source files and acquisition timestamps are
preserved, not downloaded or overwritten during a normal run.

The API snapshots are documented in `data/metadata/acquisition_manifest.json`.
The optional request example below only runs when explicitly enabled.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data/metadata/acquisition_manifest.json").is_file():
        project_root = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from within the project directory.")


import hashlib
import json

import pandas as pd
import requests
from IPython.display import display

manifest_path = project_root / "data/metadata/acquisition_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_records = {record["path"]: record for record in manifest["files"]}
if len(manifest_records) != len(manifest["files"]):
    raise ValueError("Duplicate source paths in the acquisition manifest.")


def load_snapshot(relative_path):
    """Read an unchanged source snapshot and verify its recorded checksum."""
    record = manifest_records[relative_path]
    content = (project_root / relative_path).read_bytes()
    if len(content) != record["size_bytes"]:
        raise ValueError(f"Source size mismatch: {relative_path}")
    if hashlib.sha256(content).hexdigest() != record["sha256"]:
        raise ValueError(f"Source checksum mismatch: {relative_path}")
    return json.loads(content)


data = load_snapshot("data/raw/eurostat/unemployment_poland_2024.json")
regional_data = load_snapshot("data/raw/eurostat/unemployment_nuts2_2024.json")
gdp_data = load_snapshot("data/raw/eurostat/gdp_per_capita_pps_poland_2024.json")
gdp_regional_data = load_snapshot("data/raw/eurostat/gdp_per_capita_pps_nuts2_2024.json")
nuts_reference = load_snapshot("data/raw/geography/nuts_2024_units.json")

print("Python:", sys.version.split()[0])
print("Project root:", project_root)
print("Verified source snapshots:", len(manifest_records))


## 1. API request example — optional

Parameters are visible in each manifest URL. This demonstrates `requests.get`,
a timeout, HTTP error handling and timestamps without making an online request
during normal notebook execution. Set `RUN_ONLINE_EXAMPLE = True` deliberately
to request a fresh national preview. It does not replace the analysis snapshots
or their manifest records; a refresh is a separate provenance decision.


In [ ]:
RUN_ONLINE_EXAMPLE = False

if RUN_ONLINE_EXAMPLE:
    from datetime import datetime, timezone

    source_url = manifest_records[
        "data/raw/eurostat/unemployment_poland_2024.json"
    ]["source_url"]
    request_started_at = datetime.now(timezone.utc).isoformat()
    preview_response = requests.get(source_url, timeout=30)
    response_received_at = datetime.now(timezone.utc).isoformat()
    preview_response.raise_for_status()
    preview_data = preview_response.json()
    print("HTTP status:", preview_response.status_code)
    print("Request started at:", request_started_at)
    print("Response received at:", response_received_at)
    print("Fresh preview values:", preview_data.get("value"))
else:
    print("Offline mode: using verified source snapshots.")


## 2. Confirm definitions and selected categories

Unemployment is the EU-LFS rate for ages 15–74, both sexes and all education
levels. GDP per capita is measured in PPS, not wages or disposable income.
PPP conversion is national and does not separately adjust regional prices.
The source update time is distinct from the observation year and download time.
See `docs/DATA_SOURCES.md` for definitions, source links and flag meanings.


In [ ]:
for snapshot in (regional_data, gdp_regional_data):
    print("Dataset:", snapshot["label"])
    print("Source updated:", snapshot["updated"])
    print("Dimension sizes:", snapshot["size"])
    for code in snapshot["id"]:
        if code != "geo":
            print(code, snapshot["dimension"][code]["category"]["label"])
    annotations = pd.DataFrame(
        snapshot.get("extension", {}).get("annotation", [])
    )
    if {"type", "href"}.issubset(annotations.columns):
        links = annotations.loc[annotations["type"].eq("ESMS_HTML"), "href"]
        for link in links.dropna():
            print("Reference metadata:", link)
    print()


## 3. Convert JSON-stat into regional tables

Values are matched to geographic categories by their JSON-stat positions,
not dictionary insertion order. Missing positions remain missing values.
This helper deliberately supports only a single-year slice in which `geo` is
the sole varying dimension. It rejects different filters or additional varying
dimensions instead of silently assigning observations to the wrong regions.


In [ ]:
def regional_table(payload, dataset_code, expected_categories, value_column, status_column):
    """Convert a validated single-year, single-indicator JSON-stat slice."""
    if payload.get("class") != "dataset":
        raise ValueError("Expected a JSON-stat dataset.")
    if payload.get("extension", {}).get("id", "").lower() != dataset_code.lower():
        raise ValueError("Unexpected source dataset.")
    dimension_ids = payload["id"]
    sizes = payload["size"]
    if len(dimension_ids) != len(sizes) or len(set(dimension_ids)) != len(dimension_ids):
        raise ValueError("Invalid JSON-stat dimension structure.")
    if set(dimension_ids) != {"geo", *expected_categories}:
        raise ValueError("Unexpected dimensions in the source slice.")
    for dimension_code, size in zip(dimension_ids, sizes):
        category = payload["dimension"][dimension_code]["category"]
        if dimension_code != "geo":
            if size != 1 or set(category["index"]) != {expected_categories[dimension_code]}:
                raise ValueError(f"Unexpected category or size: {dimension_code}")

    geography = payload["dimension"]["geo"]["category"]
    positions = geography["index"]
    geo_size = sizes[dimension_ids.index("geo")]
    if len(positions) != geo_size or set(positions.values()) != set(range(geo_size)):
        raise ValueError("Invalid geographic positions.")

    values = payload["value"]
    statuses = payload.get("status", {})

    def at_position(container, position):
        if isinstance(container, dict):
            return container.get(str(position))
        if isinstance(container, list) and len(container) == geo_size:
            return container[position]
        raise ValueError("Unsupported or malformed observation container.")

    rows = []
    for region_code, position in positions.items():
        rows.append({
            "region_code": region_code,
            "region_name": geography["label"][region_code],
            "year": int(expected_categories["time"]),
            value_column: at_position(values, position),
            status_column: at_position(statuses, position),
        })
    frame = pd.DataFrame(rows)
    frame[value_column] = pd.to_numeric(frame[value_column], errors="raise")
    return frame


df_unemployment_raw = regional_table(
    regional_data,
    "lfst_r_lfu3rt",
    {"freq": "A", "isced11": "TOTAL", "sex": "T", "age": "Y15-74",
     "unit": "PC", "time": "2024"},
    "unemployment_pct",
    "unemployment_status",
)

df_gdp_raw = regional_table(
    gdp_regional_data,
    "nama_10r_2gdp",
    {"freq": "A", "unit": "PPS_EU27_2020_HAB", "time": "2024"},
    "gdp_per_capita_pps",
    "gdp_status",
)

print("Unemployment source shape:", df_unemployment_raw.shape)
print("GDP source shape:", df_gdp_raw.shape)


## 4. Select the reference geography

The GISCO NUTS 2024 unit index is the authoritative code reference. Within this
index, four-character codes identify level 2. Restrict these to the 27 EU Member
States in 2024 (`EL` denotes Greece). Do not select geography by available values
or region names. Historical and non-EU categories remain in the raw snapshots.


In [ ]:
eu_country_codes_2024 = {
    "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "EL",
    "ES", "FI", "FR", "HR", "HU", "IE", "IT", "LT", "LU",
    "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK",
}

eu_nuts2_reference_codes = {
    code for code in nuts_reference
    if len(code) == 4 and code[:2] in eu_country_codes_2024
}

reference_country_codes = {code[:2] for code in eu_nuts2_reference_codes}
if reference_country_codes != eu_country_codes_2024:
    raise ValueError("Reference geography does not cover the selected EU countries.")

for label, frame in (("Unemployment", df_unemployment_raw), ("GDP", df_gdp_raw)):
    missing_codes = eu_nuts2_reference_codes - set(frame["region_code"])
    if missing_codes:
        raise ValueError(f"{label}: reference codes missing from source: {sorted(missing_codes)}")
    print(label, "categories outside reference:",
          len(set(frame["region_code"]) - eu_nuts2_reference_codes))

df_unemployment_eu_2024 = df_unemployment_raw.loc[
    df_unemployment_raw["region_code"].isin(eu_nuts2_reference_codes)
].copy()
df_gdp_eu_2024 = df_gdp_raw.loc[
    df_gdp_raw["region_code"].isin(eu_nuts2_reference_codes)
].rename(columns={"region_name": "region_name_gdp"}).copy()

print("Reference regions:", len(eu_nuts2_reference_codes))


## 5. Join and validate

Join on region code and year with one-to-one validation. Matching records do not
guarantee non-missing values. Check names, expected geography, finite values,
unemployment bounds and positive GDP before exporting. Keep observation flags.


In [ ]:
df_labour_market = df_unemployment_eu_2024.merge(
    df_gdp_eu_2024,
    on=["region_code", "year"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not df_labour_market["_merge"].eq("both").all():
    raise ValueError("Unmatched GDP records.")
if not df_labour_market["region_name"].eq(df_labour_market["region_name_gdp"]).all():
    raise ValueError("Review differing region names.")
if set(df_labour_market["region_code"]) != eu_nuts2_reference_codes:
    raise ValueError("Analysis geography differs from the reference.")

import math

for column in ("unemployment_pct", "gdp_per_capita_pps"):
    if not df_labour_market[column].dropna().map(math.isfinite).all():
        raise ValueError(f"Non-finite values in {column}.")
if not df_labour_market["unemployment_pct"].dropna().between(0, 100).all():
    raise ValueError("Unemployment values outside 0–100 percent.")
if not df_labour_market["gdp_per_capita_pps"].dropna().gt(0).all():
    raise ValueError("Non-positive GDP per capita.")

analysis_columns = [
    "region_code", "region_name", "year", "unemployment_pct",
    "unemployment_status", "gdp_per_capita_pps", "gdp_status",
]
df_analysis = df_labour_market[analysis_columns].copy()
complete_pair_mask = df_analysis[["unemployment_pct", "gdp_per_capita_pps"]].notna().all(axis=1)
low_reliability_mask = df_analysis["unemployment_status"].str.contains("u", regex=False, na=False)

quality_summary = pd.Series({
    "rows": len(df_analysis),
    "unique_regions": df_analysis["region_code"].nunique(),
    "duplicate_region_year_keys": df_analysis.duplicated(["region_code", "year"]).sum(),
    "missing_unemployment": df_analysis["unemployment_pct"].isna().sum(),
    "missing_gdp": df_analysis["gdp_per_capita_pps"].isna().sum(),
    "complete_pairs": complete_pair_mask.sum(),
    "complete_pairs_with_unemployment_u": (complete_pair_mask & low_reliability_mask).sum(),
})

display(quality_summary)
display(df_analysis.loc[~complete_pair_mask])
print("GDP observation flags:")
print(df_analysis["gdp_status"].value_counts(dropna=False))


## 6. Export derived artifacts

Only derived CSV files are rewritten. All 244 reference regions and all flags
are preserved; the next notebook selects complete pairs for correlations.
Raw snapshots and the acquisition manifest are never rewritten by this notebook.


In [ ]:
processed_dir = project_root / "data/processed"
tables_dir = project_root / "reports/tables"
processed_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

df_analysis.to_csv(
    processed_dir / "regional_labour_market_2024.csv",
    index=False,
    encoding="utf-8",
)
quality_summary.rename_axis("check").rename("value").to_csv(
    tables_dir / "data_quality_summary.csv",
    encoding="utf-8",
)
print("Saved analysis table:", df_analysis.shape)


## Verified outcome

For the preserved snapshots: 244 EU regions, 241 complete pairs and three missing
unemployment values (DEB2, PL43 and FI20), all flagged `u`. GDP is available for
all 244 regions. See `reports/tables/data_quality_summary.csv` for computed counts.

Continue with `02_exploratory_analysis.ipynb` for analysis and
`03_interactive_map.ipynb` for the interactive map.
